# 财报智析 Agent 课程作业 Notebook

作者：徐北辰（202408240218，24级智能会计班）；周亦轩（202408240302，24级智能会计班）

本 Notebook 用于展示项目运行逻辑、核心代码调用和输出结果，并附小组分工、参考资料与独立完成声明。


## 1. 研究问题

本项目研究如何构建一个会计垂直 Agent，使其能够读取上市公司年报或标准财务数据，自动完成数据校验、指标计算、风险识别和报告生成。

初步 Agent 分析流程以张新民、钱爱民《财务报表分析（第6版·立体化数字教材版）案例分析与学习指导》为权威教材依据，将教材中的偿债能力、营运能力、盈利能力、成长能力、现金流量质量和杜邦分析框架转化为可调用工具链。

问题来源于真实资本市场中的财报解读需求：深交所互动易、上证 e 互动、全景路演等平台上，投资者经常围绕经营现金流、应收账款、坏账准备、收入确认、资产负债率和财务指标变动原因向上市公司追问。相关文献也表明，年报可读性、文本复杂性和信息处理成本会影响投资者理解和风险识别。因此，本项目重点解决年报信息处理成本高、指标口径整理难、风险识别证据链弱和大模型直接生成财报结论容易幻觉的问题。

已有 CSMAR 智能财经报告分析平台、CSMAR 上市公司风险智能感知系统、弈 Chat、同花顺问财、Wind Alice、Choice 妙想 AI、Datayes AI 开放平台等相邻实践，说明 AI 财经问答、报告生成、风险诊断和 Agent 数据接口已经进入业务场景。本项目的差异是面向课堂与课程作业，强调本地可运行、确定性指标计算和 Agent trace 可追踪。

项目已完成 GitHub 仓库发布和 Streamlit Cloud 在线部署，在线演示地址：https://accounting-agent-yspsw2afdlis9itfyheqde.streamlit.app/。这一步说明系统从本地代码 Demo 推进到可访问、可交互测试、可课堂展示的初步应用状态。


In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))
PROJECT_ROOT


## 2. 读取样例数据


In [ ]:
sample_path = PROJECT_ROOT / 'data' / 'sample_financial_data.xlsx'
raw_df = pd.read_excel(sample_path)
raw_df.head()


## 3. 真实年报读取验证：科大讯飞 2025 年年度报告


In [ ]:
real_case_dir = Path('evidence') / '科大讯飞2025年报案例'
if not real_case_dir.exists():
    real_case_dir = PROJECT_ROOT / 'outputs' / 'real_case' / 'keda_xunfei_2025'

case_summary = json.loads((real_case_dir / 'real_case_summary.json').read_text(encoding='utf-8'))
case_df = pd.read_csv(real_case_dir / 'extracted_financial_data.csv')
case_summary


In [ ]:
case_df


## 4. 运行 Agent 分析主流程


In [ ]:
from agents.main_agent import run_financial_analysis
from config.settings import ModelConfig

result = run_financial_analysis(
    excel_path=sample_path,
    enable_pdf=False,
    generate_word=True,
    model_config_override=ModelConfig(enable_llm=False),
)
result.keys()


## 5. 查看数据校验结果


In [ ]:
result['validation']['summary']
result['validation']['detail_table']


## 6. 查看财务指标表


In [ ]:
result['ratio_pivot'].head(20)


## 7. 查看风险识别结果


In [ ]:
pd.DataFrame(result['risks'])


## 8. 查看 Agent 调用记录


In [ ]:
pd.DataFrame(result['trace'])


## 9. 输出文件

- Word 报告：`outputs/reports/financial_analysis_report.docx`
- Markdown 报告：`outputs/reports/financial_analysis_report.md`
- 指标表：`outputs/tables/ratio_results.xlsx`
- 数据校验表：`outputs/tables/data_validation.xlsx`
- Agent trace：`outputs/agent_trace.json`


## 10. 附录：小组贡献、参考文献与声明

### 小组贡献说明

| 成员 | 学号 | 班级 | 主要贡献 | 贡献比例 |
|---|---|---|---|---|
| 徐北辰 | 202408240218 | 24级智能会计班 | 负责智能体总体架构设计、smolagents 主控 Agent 接入、后端工具链组织、Streamlit 前端工作台构建、API 配置区、动态 Agent trace、数据读取与分析流程串联、代码调试、测试验证、GitHub 仓库发布、Streamlit Cloud 在线部署和最终工程打包。 | 60% |
| 周亦轩 | 202408240302 | 24级智能会计班 | 负责张新民、钱爱民《财务报表分析（第6版·立体化数字教材版）案例分析与学习指导》等权威教材框架、指标体系、风险规则和相关文献资料的收集整理，参与报告结构设计、页面表达优化和前端视觉美化，协助校对指标名称、报告表述和课堂展示材料。 | 40% |

### 学习心得

- 徐北辰：徐北辰在项目中主要负责把会计分析任务拆解为可执行、可追踪的 Agent 工具链。在查阅投资者互动问答和现有智能财经平台后，进一步认识到财报分析的难点不是简单生成一段结论，而是要把年报数字、指标口径、风险触发依据和报告表达连接成可复核的证据链。因此在搭建过程中，将财务数据清洗、指标计算、勾稽校验和风险识别交给确定性 Python 模块完成，让大模型承担计划生成和语言组织。通过前端工作台、动态 trace 和 Streamlit Cloud 在线部署的实现，也体会到一个可演示的 Agent 系统不仅要能算，还要能够被用户实际打开、测试和复核，让用户看清楚它正在调用什么工具、依据什么数据、输出什么结果。
- 周亦轩：周亦轩在项目中主要围绕会计专业资料和展示体验展开工作。通过整理张新民、钱爱民《财务报表分析（第6版·立体化数字教材版）案例分析与学习指导》中的教材框架、指标定义和风险规则，更加清楚地理解了传统财务报表分析中偿债能力、营运能力、盈利能力、成长能力、现金流量质量和杜邦分析之间的逻辑关系。在补充年报可读性、网络平台互动、信息处理成本和大数据审计等文献后，也进一步理解了财务报告分析工具的专业价值：它需要帮助用户降低阅读成本、统一指标口径，并把风险提示限定在审慎、可追溯的范围内。在参与前端美化和报告结构设计时，也认识到会计智能体的展示界面需要兼顾专业性和可读性，既要保持财务系统的稳重风格，也要让用户能够快速找到数据来源、指标结论、风险依据和在线演示入口。

### 教材框架与 Agent 流程对应

- 权威教材依据：张新民、钱爱民《财务报表分析（第6版·立体化数字教材版）案例分析与学习指导》
- 偿债能力分析：流动比率、速动比率、资产负债率、权益乘数。
- 营运能力分析：应收账款周转率、存货周转率、总资产周转率。
- 盈利能力分析：毛利率、净利率、ROA、ROE。
- 成长能力分析：营业收入增长率、净利润增长率。
- 现金流量质量分析：净利润现金含量、销售收现比率、利润现金流匹配性。
- 杜邦分析：ROE 分解、总资产周转率、净利率和权益乘数。

### 参考文献与资料

- 张新民、钱爱民：《财务报表分析（第6版·立体化数字教材版）案例分析与学习指导》，中国人民大学出版社，ISBN 978-7-300-31977-3。
- 谭松涛, 阚铄, 崔小勇. 互联网沟通能够改善市场信息效率吗?——基于深交所“互动易”网络平台的研究[J]. 金融研究, 2016(3):174-188.
- 徐寿福, 郑迎飞, 罗雨杰. 网络平台互动与股票异质性风险[J]. 财经研究, 2022, 48(10):153-168.
- 周波, 李洁柔, 王少飞. 整合信息披露、信息处理成本与投资意愿——基于个体投资者判断的实验研究[J]. 财经研究, 2025, 51(4):139-154.
- 徐巍, 姚振晔, 陈冬华. 中文年报可读性:衡量与检验[J]. 会计研究, 2021(03):28-44.
- 王克敏, 王华杰, 李栋栋, 戴杏云. 年报文本信息复杂性与管理者自利——来自中国上市公司的证据[J]. 管理世界, 2018, 34(12):120-132.
- 郭松林, 宁祺器, 窦斌. 上市公司年报文本增量信息与违规风险预测——基于语调和可读性的视角[J]. 统计研究, 2022, 39(12):69-84.
- 徐荣华, 朱婧, 戴欣瑜. 大数据审计:理论框架、研究进展与未来展望[J]. 外国经济与管理, 2024, 46(11):122-137.
- 深交所互动易: https://irm.cninfo.com.cn/
- 上证 e 互动: https://sns.sseinfo.com/
- 全景路演投资者关系互动平台: https://ir.p5w.net/
- CSMAR 智能财经报告分析平台: https://www.csmar.com/channels/63.html
- CSMAR 上市公司风险智能感知系统: https://www.csmar.com/channels/上市公司风险智能感知系统.html
- CSMAR 弈 Chat 智能财经对话系统: https://www.csmar.com/channels/102.html
- 同花顺问财: https://www.iwencai.com/
- Wind 金融终端与 Wind Alice: https://www.wind.com.cn/
- Choice 智能金融终端: https://choice.eastmoney.com/
- Datayes AI 开放平台: https://d.datayes.com/
- Hugging Face smolagents 文档: https://huggingface.co/docs/smolagents
- DeepSeek API 文档: https://api-docs.deepseek.com/
- Streamlit 文档: https://docs.streamlit.io/
- pandas 文档: https://pandas.pydata.org/docs/
- pdfplumber、PyMuPDF、python-docx 等开源工具文档。

### 独立完成声明

本课程作业由徐北辰、周亦轩在课程要求范围内独立完成。签名：徐北辰、周亦轩。日期：2026年6月12日。
